# EIA Weekly Petroleum Inventories

This notebook downloads weekly U.S. commercial crude oil inventory data from the 
Energy Information Administration (EIA) public API. Each observation corresponds 
to the Weekly Petroleum Status Report published every Wednesday at 10:30 AM ET. 
The notebook computes the week-on-week inventory change and classifies each report 
as bearish (inventories increased) or bullish (inventories decreased), producing 
structured news events with exact timestamps. These events serve as the primary 
news signal in the baseline statistical analysis. The dataset is saved to 
`01_data/raw/macro/eia_inventories_raw.csv`.

### General imports

In [21]:
import requests
import pandas as pd
import sqlite3

### Download data from EIA

EIA allows to download weekly oil production and inventory reports

In [24]:
url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"
params = {
    "api_key": "DEMO_KEY",
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WCRSTUS1",
    "start": "2020-01-01",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}
response = requests.get(url, params=params)
df_eia = pd.DataFrame(response.json()['response']['data'])

### Clean data

Explicit cleaning is required here because the EIA API returns a JSON response containing multiple metadata columns that are not relevant for the analysis:

- **Column selection:** The raw JSON response includes columns such as duoarea, area-name, product, process-name, series, series-description, and units. Only period (publication date) and value (inventory level in thousand barrels) are retained.

- **Type conversion:** The value column is returned as a string by the API. It is converted to numeric using pd.to_numeric(..., errors='coerce'). The errors='coerce' parameter ensures that any non-numeric values are silently converted to NaN rather than raising an exception, which prevents the pipeline from breaking on malformed entries.

- **Chronological sorting:** Records are sorted from oldest to most recent before computing weekly differences. This ordering is required to ensure that the .diff() operation produces correct sequential changes.

- **Weekly shock computation:** The week-on-week change in inventory levels is computed as inventory_change = value_this_week - value_previous_week. This delta represents the informational content of each EIA release — the market reacts not to the absolute inventory level, but to the deviation from the prior week.

- **Event direction classification:** Each weekly report is labeled according to the sign of the inventory change:

    - **bearish:** Inventories increased — signals oversupply, typically exerts downward pressure on prices \
    - **bullish:** Inventories decreased — signals scarcity, typically exerts upward pressure on prices \
    - **neutral:** No change observed

    This label serves as the structured news direction variable used in the OLS baseline analysis (*Month 1 results, p = 0.03*).
- **Exact timestamp construction:** the EIA publishes its Weekly Petroleum Status Report every Wednesday at 10:30 AM Eastern Time. An exact event timestamp is constructed by combining each report's date with this fixed publication time, then localizing it to the US/Eastern timezone. This precision is essential for correctly aligning EIA events with the hourly yfinance price data in the event window analysis.

In [25]:
df_eia = df_eia[['period', 'value']].copy()
df_eia.columns = ['date', 'inventory_mbbl']
df_eia['date'] = pd.to_datetime(df_eia['date'])
df_eia['inventory_mbbl'] = pd.to_numeric(df_eia['inventory_mbbl'], errors='coerce')
df_eia = df_eia.sort_values('date').reset_index(drop=True)

df_eia['inventory_change'] = df_eia['inventory_mbbl'].diff()
df_eia['news_direction'] = df_eia['inventory_change'].apply(
    lambda x: 'bearish' if x > 0 else ('bullish' if x < 0 else 'neutral')
)
df_eia['datetime_et'] = pd.to_datetime(df_eia['date']) + pd.Timedelta(hours=10, minutes=30)
df_eia['datetime_et'] = df_eia['datetime_et'].dt.tz_localize('US/Eastern')
df_eia['datetime_et'] = df_eia['datetime_et'].dt.tz_convert('UTC')
df_eia['datetime_et'] = df_eia['datetime_et'].dt.ceil('h')  # 15:30 → 16:00

print(f"Entries: {len(df_eia)}")
print(f"Range: {df_eia['date'].min()} a {df_eia['date'].max()}")
print(f"Bearish: {(df_eia['news_direction']=='bearish').sum()}")
print(f"Bullish: {(df_eia['news_direction']=='bullish').sum()}")
print(df_eia.tail(5))


Entries: 331
Range: 2020-01-03 00:00:00 a 2026-05-01 00:00:00
Bearish: 139
Bullish: 191
          date  inventory_mbbl  inventory_change news_direction  \
326 2026-04-03          878042            1342.0        bearish   
327 2026-04-10          872985           -5057.0        bullish   
328 2026-04-17          870774           -2211.0        bullish   
329 2026-04-24          857419          -13355.0        bullish   
330 2026-05-01          849882           -7537.0        bullish   

                  datetime_et  
326 2026-04-03 15:00:00+00:00  
327 2026-04-10 15:00:00+00:00  
328 2026-04-17 15:00:00+00:00  
329 2026-04-24 15:00:00+00:00  
330 2026-05-01 15:00:00+00:00  


### Save to DB

In [26]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_eia['date'] = df_eia['date'].astype(str)
df_eia['datetime_et'] = df_eia['datetime_et'].astype(str)

df_eia.to_sql('eia_events', conn, if_exists='replace', index=False)

print(f"EIA records saved: {len(df_eia)}")
print(pd.read_sql("""
    SELECT MIN(date) as earliest, MAX(date) as latest, 
           COUNT(*) as total
    FROM eia_events
""", conn))

# Verify datetime_et format
print(pd.read_sql("SELECT datetime_et FROM eia_events LIMIT 3", conn))

conn.close()

EIA records saved: 331
     earliest      latest  total
0  2020-01-03  2026-05-01    331
                 datetime_et
0  2020-01-03 16:00:00+00:00
1  2020-01-10 16:00:00+00:00
2  2020-01-17 16:00:00+00:00


### Merge EIA entries into market_context

In [27]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_eia = pd.read_sql("SELECT inventory_change, datetime_et FROM eia_events", conn)
df_eia['datetime_et'] = pd.to_datetime(df_eia['datetime_et'], utc=True)
df_eia = df_eia.sort_values('datetime_et').reset_index(drop=True)

df_market = pd.read_sql("SELECT * FROM market_context", conn)
df_market['datetime_hour'] = pd.to_datetime(df_market['datetime_hour'], utc=True)
df_market = df_market.sort_values('datetime_hour').reset_index(drop=True)

df_merged = pd.merge_asof(
    df_market,
    df_eia[['datetime_et', 'inventory_change']],
    left_on='datetime_hour',
    right_on='datetime_et',
    direction='backward'
)

df_merged = df_merged.rename(columns={'inventory_change': 'eia_surprise'})
df_merged['eia_surprise'] = df_merged['eia_surprise'].fillna(0)
df_merged['is_eia_release'] = (df_merged['datetime_et'] == df_merged['datetime_hour']).astype(int)
df_merged = df_merged.drop(columns=['datetime_et'])

df_merged['datetime_hour'] = df_merged['datetime_hour'].astype(str)
df_merged.to_sql('market_context', conn, if_exists='replace', index=False)

print("Done")
print(pd.read_sql("""
    SELECT datetime_hour, eia_surprise, is_eia_release
    FROM market_context
    WHERE datetime_hour >= '2024-05-15'
    AND datetime_hour <= '2024-05-17'
    ORDER BY datetime_hour
""", conn))

conn.close()

Done
                datetime_hour  eia_surprise  is_eia_release
0   2024-05-15 00:00:00+00:00       -1915.0               0
1   2024-05-15 01:00:00+00:00       -1915.0               0
2   2024-05-15 02:00:00+00:00       -1915.0               0
3   2024-05-15 03:00:00+00:00       -1915.0               0
4   2024-05-15 04:00:00+00:00       -1915.0               0
5   2024-05-15 05:00:00+00:00       -1915.0               0
6   2024-05-15 06:00:00+00:00       -1915.0               0
7   2024-05-15 07:00:00+00:00       -1915.0               0
8   2024-05-15 08:00:00+00:00       -1915.0               0
9   2024-05-15 09:00:00+00:00       -1915.0               0
10  2024-05-15 10:00:00+00:00       -1915.0               0
11  2024-05-15 11:00:00+00:00       -1915.0               0
12  2024-05-15 12:00:00+00:00       -1915.0               0
13  2024-05-15 13:00:00+00:00       -1915.0               0
14  2024-05-15 14:00:00+00:00       -1915.0               0
15  2024-05-15 15:00:00+00:00      